In [37]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [38]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

In [39]:
input_shape = (128, 128, 3)
batch_size = 32
epochs_stage1 = 10
epochs_stage2 = 10

patch_size = 16
projection_dim = 64
transformer_layers = 4
num_heads = 8
num_classes = 8

In [40]:
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "/kaggle/input/datasets/amimulahasanrofik/alz-b-1100/alzheimer_new_11/alzheimer_new",
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(128, 128),
    batch_size=batch_size
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "/kaggle/input/datasets/amimulahasanrofik/alz-b-1100/alzheimer_new_11/alzheimer_new",
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(128, 128),
    batch_size=batch_size
)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

Found 8800 files belonging to 8 classes.
Using 7040 files for training.
Found 8800 files belonging to 8 classes.
Using 1760 files for validation.


In [41]:
# class Patches(layers.Layer):
#     def __init__(self, patch_size):
#         super().__init__()
#         self.patch_size = patch_size

#     def call(self, images):
#         # Ensure shape is static
#         batch_size = tf.shape(images)[0]
#         patch_h = images.shape[1] // self.patch_size
#         patch_w = images.shape[2] // self.patch_size

#         patches = tf.image.extract_patches(
#             images=images,
#             sizes=[1, self.patch_size, self.patch_size, 1],
#             strides=[1, self.patch_size, self.patch_size, 1],
#             padding="VALID",
#         )
#         patches = tf.reshape(patches, [batch_size, patch_h * patch_w, -1])
#         return patches

In [42]:
# class PatchEncoder(layers.Layer):
#     def __init__(self, num_patches, projection_dim):
#         super().__init__()
#         self.projection = layers.Dense(projection_dim)
#         self.position_embedding = layers.Embedding(
#             input_dim=num_patches, output_dim=projection_dim
#         )

#     def call(self, patches):
#         positions = tf.range(start=0, limit=tf.shape(patches)[1])
#         encoded = self.projection(patches) + self.position_embedding(positions)
#         return encoded

In [43]:
class Patches(layers.Layer):
    def __init__(self, patch_size):
        super().__init__()
        self.patch_size = patch_size

    def call(self, images):
        batch_size = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            padding="VALID",
        )
        patch_dim = patches.shape[-1]
        patches = tf.reshape(patches, [batch_size, -1, patch_dim])
        return patches

In [44]:
class PatchEncoder(layers.Layer):
    def __init__(self, num_patches, projection_dim):
        super().__init__()
        self.projection = layers.Dense(projection_dim)
        self.position_embedding = layers.Embedding(num_patches, projection_dim)

    def call(self, patches):
        positions = tf.range(start=0, limit=tf.shape(patches)[1], delta=1)
        return self.projection(patches) + self.position_embedding(positions)

In [45]:
def build_model():
    inputs = layers.Input(shape=input_shape)

    # Data Augmentation
    data_aug = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
    ])
    x_aug = data_aug(inputs)

    # CNN Branch — ResNet50
    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_tensor=x_aug
    )
    base_model.trainable = False
    x_cnn = layers.GlobalAveragePooling2D()(base_model.output)

    # ViT Branch
    num_patches = (input_shape[0] // patch_size) ** 2
    x_scaled = layers.Rescaling(1./255)(x_aug)
    patches = Patches(patch_size)(x_scaled)
    encoded = PatchEncoder(num_patches, projection_dim)(patches)

    for _ in range(transformer_layers):
        x1 = layers.LayerNormalization()(encoded)
        attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim)(x1, x1)
        x2 = layers.Add()([attn, encoded])
        x3 = layers.LayerNormalization()(x2)
        encoded = layers.Add()([x3, x2])

    x_vit = layers.LayerNormalization()(encoded)
    x_vit = layers.GRU(256)(x_vit)

    # Merge
    merged = layers.concatenate([x_cnn, x_vit])
    merged = layers.Dense(256, activation='relu')(merged)
    merged = layers.Dropout(0.3)(merged)
    outputs = layers.Dense(num_classes, activation='softmax')(merged)

    model = keras.Model(inputs, outputs)
    return model, base_model

In [46]:
model, base_model = build_model()

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(patience=3),
    keras.callbacks.ModelCheckpoint("best_model.h5", save_best_only=True)
]

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs_stage1,
    callbacks=callbacks
)

TypeError: Exception encountered when calling Patches.call().

[1mCould not automatically infer the output shape / dtype of 'patches_5' (of type Patches). Either the `Patches.call()` method is incorrect, or you need to implement the `Patches.compute_output_spec() / compute_output_shape()` method. Error encountered:

Missing required positional argument[0m

Arguments received by Patches.call():
  • args=('<KerasTensor shape=(None, 128, 128, 3), dtype=float32, sparse=False, ragged=False, name=keras_tensor_1150>',)
  • kwargs=<class 'inspect._empty'>

In [ ]:
for layer in base_model.layers[-30:]:
    layer.trainable = True

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs_stage2,
    callbacks=callbacks
)

In [ ]:
test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "/kaggle/input/datasets/amimulahasanrofik/bra-alz-8/alzheimer_new",
    image_size=(128, 128),
    batch_size=batch_size,
    shuffle=False
)

In [ ]:
y_true = np.concatenate([y for x, y in test_ds], axis=0)
y_pred = model.predict(test_ds)
y_pred_labels = np.argmax(y_pred, axis=1)

cm = confusion_matrix(y_true, y_pred_labels)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

print(classification_report(y_true, y_pred_labels))